# Week3_ex4 - Induced Current of Coil

Same setup as ex3 (ring coil next to a straight wire, EddyCurrent solve) but this time pulling out flux, emf, and the induced current in the ring together, all plotted against phase.

In [ ]:
import pandas as pd
import numpy as np
import os
import ansys.aedt.core
import math
import shutil

In [ ]:
# 1. AEDT desktop interface
DT = ansys.aedt.core.Desktop(version="2025.2", non_graphical=False, student_version=True)

# 2. turn off autosave
DT.disable_autosave()

# 3. create project and set solution type
sol_type = "EddyCurrent"
M3D = ansys.aedt.core.maxwell.Maxwell3d(solution_type=sol_type)

# 4. odesign object for script-recording style API calls
oDesign = M3D.odesign


In [ ]:
## Set up output directory for results ##

proj_name = "Week3_ex4"

# If ANSYS_PROJECT_DIR is set on this machine, save there.
# Otherwise, fall back to the notebook's own working directory,
# so this stays portable for anyone else running the notebook as-is.
base_dir = os.environ.get("ANSYS_PROJECT_DIR", os.getcwd())
dir = os.path.join(base_dir, proj_name)
print(dir)
os.makedirs(dir, exist_ok=True)

desi_name = "Week3_ex4"

## Save project and apply design name ##

proj = M3D.oproject

proj.SaveAs(f"{dir}\\{proj_name}.aedt", True)
M3D.rename_design(desi_name, save=False)

# Save again so the design-name change is committed to disk
M3D.save_project()


In [ ]:
# Helper to find the two terminal faces of a wire/coil object
# Looks at face center x-coordinates and picks the two faces with the largest |x| (the wire ends)
def find_terminal_face(winding_obj) :
    terminal_face = []

    # find maximum x position among all faces
    max_x_pos = max(abs(face.center[0]) for face in winding_obj.faces)

    # collect the faces sitting at that max x position
    for face in winding_obj.faces :
        if abs(abs(face.center[0]) - max_x_pos) <= 0.0001 :
            terminal_face.append(face)

    # sort by x so we consistently get (positive-x face, negative-x face)
    ter_out, ter_in = sorted(terminal_face, key=lambda x : x.center[0], reverse=True)

    return ter_out, ter_in


In [ ]:
line_length = 400
M3D["line_length"] = f"{line_length}mm"

line_diameter = 10
M3D["line_diameter"] = f"{line_diameter}mm"

line_num_seg = 36
M3D["line_num_seg"] = f"{line_num_seg}"

Current = 100
M3D["Current"] = f"{Current}A"

ring_dis = 200
M3D["ring_dis"] = f"{ring_dis}mm"

coil_diameter = 10
M3D["coil_diameter"] = f"{coil_diameter}mm"


In [ ]:
# Setup created via script-recording style call (bypasses pyaedt's create_setup / M3D.setups[-1],
# which raise KeyError: 'ac magnetic' on AEDT 2025 R2 due to the EddyCurrent -> AC Magnetic rename)
setup_name = "Setup1"
oModule = oDesign.GetModule("AnalysisSetup")

oModule.InsertSetup("EddyCurrent", 
	[
		"NAME:Setup1",
		"Enabled:="		, True,
		[
			"NAME:MeshLink",
			"ImportMesh:="		, False
		],
		"MaximumPasses:="	, 20,
		"MinimumPasses:="	, 2,
		"MinimumConvergedPasses:=", 1,
		"PercentRefinement:="	, 30,
		"SolveFieldOnly:="	, False,
		"PercentError:="	, 1,
		"SolveMatrixAtLast:="	, True,
		"UseNonLinearIterNum:="	, False,
		"UseIterativeSolver:="	, False,
		"RelativeResidual:="	, 1E-05,
		"NonLinearResidual:="	, 0.0001,
		"RelaxationFactor:="	, 1,
		"SmoothBHCurve:="	, False,
		"Frequency:="		, "10Hz",
		"HasSweepSetup:="	, False,
		"UseHighOrderShapeFunc:=", False,
		"ImportMeshForMuLink:="	, False,
		"LossAdaptiveCtrl:="	, "0.5",
		"UseMuLink:="		, False
	])


In [ ]:
# 1. draw the straight wire
point_tmp = []
point_tmp.append(["-line_length/2", "0mm", "0mm"])
point_tmp.append(["line_length/2", "0mm", "0mm"])
line = M3D.modeler.create_polyline(points=point_tmp, name="line", material="copper",
                            xsection_type="Circle", xsection_width=line_diameter, xsection_num_seg=line_num_seg)


# 2. draw the ring coil (two cylinders, subtract the inner one from the outer one)
center = [0, "-ring_dis", "-0.5mm"]
origin = [0, "-ring_dis+coil_diameter+0.5mm", "-0.5mm"]
ring = M3D.modeler.create_polyhedron(orientation="Z", center=center, origin=origin, height="1mm", num_sides=24, name="ring", material="copper")

center = [0, "-ring_dis", "-0.5mm"]
origin = [0, "-ring_dis+coil_diameter-0.5mm", "-0.5mm"]
tmp = M3D.modeler.create_polyhedron(orientation="Z", center=center, origin=origin, height="1mm", num_sides=24, name=None, material=None)
M3D.modeler.subtract(blank_list=ring, tool_list=tmp, keep_originals=False)


# 3. create region and assign radiation boundary
pad_value = ["line_length/2", "-line_length/2", "line_length*3", "-line_length*3", "line_length*3", "-line_length*3"]
region = M3D.modeler.create_region(pad_value=pad_value, pad_type='Absolute Position', name='Region')

assignment = [region.top_face_y, region.bottom_face_y, region.top_face_z, region.bottom_face_z]

# NOTE: M3D.assign_radiation() checks that solution_type == "EddyCurrent" exactly and
# raises AEDTRuntimeError("Excitation applicable only to Eddy Current.") otherwise. But
# AEDT 2025 R2 renamed this solver "AC Magnetic" internally, so self.solution_type reports
# "AC Magnetic" here -- the check fails even though this IS an eddy current solve. Bypassing
# the pyaedt wrapper and calling the raw AEDT boundary API directly sidesteps this check.
oModule = oDesign.GetModule("BoundarySetup")
face_ids = [f.id for f in assignment]
oModule.AssignRadiation(
	[
		"NAME:Radiation1",
		"Objects:=", [],
		"Faces:=", face_ids
	])


# 4. draw the surfaces/lines used to pull field data
origin = [0, "-ring_dis", 0]
field_circle = M3D.modeler.create_circle(orientation="XY", origin=origin, radius="coil_diameter", 
                                           num_sides=0, is_covered=True, name="field_circle", material=None, non_model=False)  # surface for linkage flux


origin = [0, "-ring_dis", 0]
cs1 = M3D.modeler.create_coordinate_system(origin=origin, reference_cs='Global', name="cs1")
M3D.modeler.set_working_coordinate_system(name=cs1)  # switch to a coordinate system centered on the coil

M3D.modeler.section(assignment=ring, plane="YZ", create_new=True, section_cross_object=False)
ring_sheet = M3D.modeler.sheet_objects[-1]  # surface for induced current
ring_sheet.name = "ring_sheet"

M3D.modeler.split(assignment=ring_sheet, plane="ZX", sides="NegativeOnly")
M3D.modeler.set_working_coordinate_system(name="Global")  # switch back to the global coordinate system


field_circle_line = M3D.modeler.create_object_from_edge(assignment=field_circle.edges, non_model=False)  # line for induced voltage
field_circle_line.name = "field_circle_line"



In [ ]:
oModule = oDesign.GetModule("MeshSetup")

oModule.AssignLengthOp(
	[
		"NAME:ring_mesh",
		"RefineInside:="	, True,
		"Enabled:="		, True,
		"Objects:="		, ["ring"],
		"RestrictElem:="	, False,
		"NumMaxElem:="		, "3000",
		"RestrictLength:="	, True,
		"MaxLength:="		, "coil_diameter/10",   # 0.5mm → 1mm
	])

oModule.AssignLengthOp(
[
	"NAME:line_mesh",
	"RefineInside:="	, True,
	"Enabled:="		, True,
	"Objects:="		, ["line"],
	"RestrictElem:="	, False,
	"NumMaxElem:="		, "3000",
	"RestrictLength:="	, True,
	"MaxLength:="		, "line_diameter",        # 5mm → 10mm
])

In [ ]:
# 1. set the wire's two end faces as coil terminals
ter_out, ter_in = find_terminal_face(line)

M3D.assign_coil(assignment=ter_out, conductors_number=1, polarity="Negative", name="Out")
M3D.assign_coil(assignment=ter_in, conductors_number=1, polarity="Positive", name="In")


# 2. create a winding and add the coils to it
# NOTE: assign_coil/assign_winding/add_winding_coils are EddyCurrent-only pyaedt calls that
# haven't been locally verified yet against the AC Magnetic rename issue (see assign_radiation
# above). If any of these throw a KeyError/AEDTRuntimeError mentioning "AC Magnetic" or
# "Eddy Current", swap to the equivalent raw oDesign.GetModule(...) call instead.
coil = M3D.assign_winding(assignment=None, winding_type='Current', is_solid=True, current="Current", 
                   resistance=0, inductance=0, voltage=0, parallel_branches=1, phase=0, 
                   name="coil")

M3D.add_winding_coils(coil.name, coils=["Out", "In"])


In [ ]:
oModule = oDesign.GetModule("FieldsReporter")

# H field linked through the ring
oModule.AddNamedExpression("ring_H", "Fields", 
	[
		"NameOfExpression:="	, ["Vector_H"],
		"Operation:="		, ["Normal"],
		"Operation:="		, ["Dot"],
		"EnterSurface:="	, ["field_circle"],
		"Operation:="		, ["SurfaceValue"],
		"Operation:="		, ["Integrate"]
	])

# B field linked through the ring
oModule.AddNamedExpression("ring_B", "Fields", 
	[
		"NameOfExpression:="	, ["Vector_B"],
		"Operation:="		, ["Normal"],
		"Operation:="		, ["Dot"],
		"EnterSurface:="	, ["field_circle"],
		"Operation:="		, ["SurfaceValue"],
		"Operation:="		, ["Integrate"]
	])

# induced current in the ring
oModule.AddNamedExpression("ring_current", "Fields", 
	[
		"NameOfExpression:="	, ["Vector_J"],
		"Operation:="		, ["Normal"],
		"Operation:="		, ["Dot"],
		"EnterSurface:="	, ["ring_sheet"],
		"Operation:="		, ["SurfaceValue"],
		"Operation:="		, ["Integrate"]
	])

# induced emf in the ring
oModule.AddNamedExpression("ring_emf", "Fields", 
	[
		"NameOfExpression:="	, ["Vector_E"],
		"Operation:="		, ["Tangent"],
		"Operation:="		, ["Dot"],
		"EnterLine:="		, ["field_circle_line"],
		"Operation:="		, ["LineValue"],
		"Operation:="		, ["Integrate"]
	])



In [ ]:
oDesign.Analyze(setup_name)


In [ ]:
oModule = oDesign.GetModule("ReportSetup")

oModule.CreateReport("Calculator Expressions Plot 1", "Fields", "Rectangular Stacked Plot", "Setup1 : LastAdaptive", [], 
	[
		"Phase:="		, ["All"],
		"Freq:="		, ["All"],
		"line_length:="		, ["Nominal"],
		"line_diameter:="	, ["Nominal"],
		"line_num_seg:="	, ["Nominal"],
		"Current:="		, ["Nominal"],
		"ring_dis:="		, ["Nominal"],
		"coil_diameter:="	, ["Nominal"]
	], 
	[
		"X Component:="		, "Phase",
		"Y Component:="		, ["ring_B","ring_emf", "ring_current"]
	])

csv_name = "Week3_ex4_result.csv"
dir_csv = csv_name  # saved flat, next to the notebook -- same convention as Images/
oModule.ExportToFile("Calculator Expressions Plot 1", dir_csv, False)


In [ ]:
## plot results in python ##

import numpy as np
import matplotlib.pyplot as plt

# 1) read the exported CSV
df = pd.read_csv(dir_csv)

# 2) pull out the columns we need
phase_deg  = df['Phase [deg]'].values
flux_uWb   = df['ring_B []'].values * 1e6         # Wb -> uWb
emf_uV     = df['ring_emf []'].values * 1e6       # V  -> uV
current_mA = df['ring_current []'].values * 1e3   # A  -> mA

# 3) stacked plot (top: flux, middle: emf, bottom: current)
fig, axes = plt.subplots(3, 1, sharex=True, figsize=(8, 6), constrained_layout=True)

# --- flux plot ---
axes[0].plot(phase_deg, flux_uWb, color='tab:blue')
axes[0].set_ylabel('Flux (uWb)')
axes[0].grid(True, alpha=0.3)

i_flux_max = int(np.nanargmax(flux_uWb))
x_flux_max = phase_deg[i_flux_max]
y_flux_max = flux_uWb[i_flux_max]
axes[0].plot(x_flux_max, y_flux_max, 'o', color='tab:blue')
axes[0].annotate(
    f"max: {y_flux_max:.3g} uWb\n@ {x_flux_max:.3g} deg",
    xy=(x_flux_max, y_flux_max),
    xytext=(10, 10),
    textcoords='offset points',
    arrowprops=dict(arrowstyle='->', color='tab:blue', lw=1),
    fontsize=9,
    color='tab:blue',
    ha='left',
    va='bottom',
    bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='tab:blue', alpha=0.85),
)

# --- emf plot ---
axes[1].plot(phase_deg, emf_uV, color='tab:red')
axes[1].set_ylabel('EMF (uV)')
axes[1].grid(True, alpha=0.3)

i_emf_max = int(np.nanargmax(emf_uV))
x_emf_max = phase_deg[i_emf_max]
y_emf_max = emf_uV[i_emf_max]
axes[1].plot(x_emf_max, y_emf_max, 'o', color='tab:red')
axes[1].annotate(
    f"max: {y_emf_max:.3g} uV\n@ {x_emf_max:.3g} deg",
    xy=(x_emf_max, y_emf_max),
    xytext=(10, 10),
    textcoords='offset points',
    arrowprops=dict(arrowstyle='->', color='tab:red', lw=1),
    fontsize=9,
    color='tab:red',
    ha='left',
    va='bottom',
    bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='tab:red', alpha=0.85),
)

# --- current plot ---
axes[2].plot(phase_deg, current_mA, color='tab:green')
axes[2].set_xlabel('Phase (deg)')
axes[2].set_ylabel('Current (mA)')
axes[2].grid(True, alpha=0.3)

i_cur_max = int(np.nanargmax(current_mA))
x_cur_max = phase_deg[i_cur_max]
y_cur_max = current_mA[i_cur_max]
axes[2].plot(x_cur_max, y_cur_max, 'o', color='tab:green')
axes[2].annotate(
    f"max: {y_cur_max:.3g} mA\n@ {x_cur_max:.3g} deg",
    xy=(x_cur_max, y_cur_max),
    xytext=(10, 10),
    textcoords='offset points',
    arrowprops=dict(arrowstyle='->', color='tab:green', lw=1),
    fontsize=9,
    color='tab:green',
    ha='left',
    va='bottom',
    bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='tab:green', alpha=0.85),
)

os.makedirs("Images", exist_ok=True)
plt.savefig("Images/Week3_ex4_flux_emf_current_vs_phase.png", dpi=150)
plt.show()


In [ ]:
## save project ##

M3D.save_project()
